# Crea tu video de Lyrics o Karaoke

**Antes de empezar**
- Ten a mano un audio, un video o un enlace de YouTube propio o autorizado.
- Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU* (su disponibilidad depende de Colab).

**Pasos:** Iniciar → Elegir canción → Datos → Preparar → Generar → Descargar. La corrección de letra (4b) y el fondo (4c) son opcionales.

**Cómo usar el cuaderno**
- Ejecuta cada celda en orden con ▶. No uses «Ejecutar todas».
- En el paso 2 elige solo **2a o 2b**.
- Celdas con opciones: elige antes de pulsar ▶.
- Celdas de subida: al ejecutar se abre el selector de archivos.
- Pasos 3 y 4c.1: ejecuta primero y edita los controles que aparecen debajo.

Todo vive en esta sesión temporal de Colab (no se usa Google Drive): descarga tu video antes de cerrarla.


In [ ]:
# @title 1 · Iniciar
# @markdown Instala el motor y comprueba la GPU.
import json
import os
import shutil
import signal
import subprocess
import sys
import uuid
import zipfile
from hashlib import sha256
from html import escape
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlopen

import ipywidgets as widgets
from google.colab import files, output
from IPython.display import Video, display

Modelo_Whisper = 'large-v3 (más preciso)' # @param ['small (rápido)', 'medium (equilibrado)', 'large-v3 (más preciso)']

if sys.version_info < (3, 12):  # noqa: UP036 - validate the host before installing the package
    raise RuntimeError('Selecciona un entorno de Colab con Python 3.12 o posterior.')
ROOT = Path('/content/karaoke-notebook')
ROOT.mkdir(exist_ok=True)
OUTPUT_DIR = ROOT / 'projects'
MODEL_CACHE = ROOT / 'models'
WHISPER_MODEL = Modelo_Whisper.split()[0]
ENGINE = Path('/content/karaoke-env/bin/python')
RELEASE_VERSION = '0.1.12'
RELEASE_SHA256 = '22303afb91ba6a9bacf04abda456d2b52b6ec12c5ca3b76d249034eb1329841a'

def download_release_wheel(folder):
    """Download the selected public release and verify its SHA-256 before installation."""
    folder.mkdir(parents=True, exist_ok=True)
    wheel = folder / f'lyrics_generator-{RELEASE_VERSION}-py3-none-any.whl'
    base = f'https://github.com/Fer-Bar/word-sync-generator/releases/download/v{RELEASE_VERSION}'
    local_wheel = Path.cwd() / wheel.name
    if local_wheel.is_file():
        content = local_wheel.read_bytes()
    else:
        print(f'Descargando motor {RELEASE_VERSION} desde GitHub Releases…', flush=True)
        try:
            with urlopen(f'{base}/{wheel.name}', timeout=60) as response:
                content = response.read()
        except HTTPError as error:
            if error.code != 404:
                raise RuntimeError('No se pudo descargar el motor. Revisa la conexión y repite el paso 1.') from error
            print(f'Esta entrega aún no está publicada. Sube {wheel.name} de la misma carpeta que el cuaderno.')
            uploaded = files.upload()
            if set(uploaded) != {wheel.name}:
                raise RuntimeError(f'Selecciona únicamente {wheel.name} y repite el paso 1.') from error
            content = uploaded[wheel.name]
        except OSError as error:
            raise RuntimeError('No se pudo descargar el motor. Revisa la conexión y repite el paso 1.') from error
    if sha256(content).hexdigest() != RELEASE_SHA256:
        raise RuntimeError('Este wheel no corresponde al cuaderno. Usa los dos archivos de la misma entrega.')
    wheel.write_bytes(content)
    return wheel

def run_command(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError('El proceso no terminó correctamente. Revisa el mensaje anterior; no continúes.')
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise

def run_engine(operation, request):
    request_path = ROOT / f'{uuid.uuid4().hex}.request.json'
    result_path = request_path.with_suffix('.result.json')
    request_path.write_text(json.dumps(request), encoding='utf-8')
    run_command([str(ENGINE), '-u', '-m', 'karaoke_app.notebook_runner', operation,
                 '--request', str(request_path), '--result', str(result_path)])
    if not result_path.is_file():
        raise RuntimeError('No se recibió un resultado completo. Revisa los mensajes del proceso.')
    return json.loads(result_path.read_text(encoding='utf-8'))

def song_snapshot(form):
    state, fields = form['state'], form['fields']
    lookup = state['reference'] is None and (fields['search'].value if 'search' in fields else True)
    return (str(state['audio']), str(state['reference']), WHISPER_MODEL, lookup,
            *[fields[key].value.strip() for key in ('title', 'artist', 'album', 'language')])

def ensure_current_song():
    if not globals().get('prepared'):
        raise RuntimeError('Primero completa el paso 4.')
    form = globals().get('song_form', {})
    state = form.get('state', {})
    if (state.get('prepared_directory') != prepared['directory']
            or state.get('prepared_snapshot') != song_snapshot(form)):
        raise RuntimeError('Cambiaste los datos de la canción. Repite el paso 4 o vuelve a los valores anteriores.')

wheel = download_release_wheel(ROOT / 'package' / uuid.uuid4().hex)
with zipfile.ZipFile(wheel) as package:
    requirements = package.read('karaoke_app/runtime/requirements.txt')
requirements_path = ROOT / 'requirements.txt'
requirements_path.write_bytes(requirements)
uv = shutil.which('uv')
if uv is None:
    bootstrap = ROOT / 'bootstrap'
    run_command([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap), 'uv==0.13.4'])
    uv = str(bootstrap / 'bin' / 'uv')
run_command([uv, 'venv', '--allow-existing', '--python', sys.executable, str(ENGINE.parent.parent)])
run_command([uv, 'pip', 'install', '--python', str(ENGINE), '--reinstall-package', 'lyrics-generator',
             str(wheel), '-r', str(requirements_path)])
run_command([uv, 'pip', 'check', '--python', str(ENGINE)])
print('Instalación completa.')

# Los controles viven en el Python de Colab; el motor sigue aislado.
output.enable_custom_widget_manager()

def create_song_form():
    """Create separate uploads and metadata controls for the current session."""
    fields = {
        'title': widgets.Text(description='Título', placeholder='Nombre real de la canción'),
        'artist': widgets.Text(description='Artista', placeholder='Intérprete de esta versión'),
        'album': widgets.Text(description='Álbum', placeholder='Opcional'),
        'language': widgets.Dropdown(description='Idioma', options=[
            ('Selecciona el idioma', ''), *[tuple(item) for item in report['notebook_languages']]]),
    }
    for field in fields.values():
        field.layout = widgets.Layout(width='100%', max_width='520px')
        field.style.description_width = '70px'
    state = {'audio': None, 'reference': None, 'metadata': {}, 'language_manual': False}
    lyrics_upload = widgets.FileUpload(
        accept='.txt', multiple=False, description='Subir letra TXT',
        tooltip='Opcional. Sin un TXT se busca la letra automáticamente.', layout=widgets.Layout(width='190px'))
    lyrics_label = widgets.HTML(layout=widgets.Layout(display='none', width='100%'))
    remove = widgets.Button(description='Quitar', icon='times',
                            layout=widgets.Layout(width='100px', display='none'))
    notice = widgets.HTML(layout=widgets.Layout(display='none', width='100%'))

    def refresh_lyrics():
        reference = state['reference']
        remove.layout.display = '' if reference else 'none'
        lyrics_label.layout.display = '' if reference else 'none'
        lyrics_upload.description = 'Cambiar TXT' if reference else 'Subir letra TXT'
        if reference:
            lyrics_label.value = (f'<span style="overflow-wrap:anywhere">✓ <b>{escape(reference.name)}</b>'
                                  f' · {reference.stat().st_size / 1024:.1f} KB recibidos</span>')
        else:
            lyrics_label.value = ''

    def invalidate(change=None):
        if globals().get('song_form', {}).get('state') is not state:
            return
        state['inputs_changed'] = (state.get('prepared_snapshot') is not None
                                   and state['prepared_snapshot'] != song_snapshot(song_form))
        notice.value = ''
        notice.layout.display = 'none'
        refresh_lyrics()
        notice.value = ('Cambiaste los datos. Repite el paso 4 o vuelve a los valores anteriores.'
                        if state['inputs_changed'] else '')
        notice.layout.display = '' if state['inputs_changed'] else 'none'

    def receive(change):
        if not change['new']:
            return
        try:
            value = change['new']
            if isinstance(value, dict):  # ipywidgets 7; la versión 8 entrega una tupla.
                entries = [dict(item, name=name) for name, item in value.items()]
            else:
                entries = list(value)
            if len(entries) != 1:
                raise ValueError('Elige exactamente un archivo en este botón.')
            item = entries[0]
            name = Path(item['name']).name
            if Path(name).suffix.lower() != '.txt':
                raise ValueError('Selecciona un archivo de letra .txt.')
            if not len(item['content']):
                raise ValueError('El archivo está vacío. Selecciona otro.')
            folder = ROOT / 'uploads' / uuid.uuid4().hex
            folder.mkdir(parents=True)
            path = folder / name
            path.write_bytes(bytes(item['content']))
            state['reference'] = path
            invalidate()
        except (OSError, ValueError, RuntimeError) as error:
            notice.value = f'<b>No se pudo cargar el TXT.</b> {escape(str(error))}'
            notice.layout.display = ''
        finally:
            # No conservar el contenido del archivo en el estado guardable del widget.
            lyrics_upload.value = {} if isinstance(lyrics_upload.value, dict) else ()

    lyrics_upload.observe(receive, names='value')

    def clear_lyrics(button):
        state['reference'] = None
        invalidate()

    remove.on_click(clear_lyrics)
    def language_changed(change):
        if not state.get('setting_language'):
            state['language_manual'] = bool(change['new'])

    fields['language'].observe(language_changed, names='value')
    for field in fields.values():
        field.observe(invalidate, names='value')
    reference_panel = widgets.VBox([
        widgets.HBox([lyrics_upload, remove], layout=widgets.Layout(
            flex_flow='row wrap', align_items='center', gap='6px')),
        lyrics_label, notice,
    ], layout=widgets.Layout(width='100%', max_width='520px', gap='2px', margin='8px 0 0 0'))
    refresh_lyrics()
    panel = widgets.VBox([
        widgets.HTML('<b>Datos de la canción</b>'),
        fields['title'], fields['artist'], fields['album'], fields['language'],
        reference_panel,
    ], layout=widgets.Layout(width='100%', max_width='600px', padding='12px', gap='8px'))
    return {'panel': panel, 'fields': fields, 'state': state, 'refresh': invalidate}

prepared = rendered = None
report = run_engine('diagnose', {})
if (report.get('engine_version') != RELEASE_VERSION
        or not {'youtube', 'configure', 'detect-language', 'inspect-design'}.issubset(
            report.get('notebook_features', []))):
    raise RuntimeError('Paquete incompatible. Usa el cuaderno de la misma release y repite el paso 1.')
if not report['cuda_available']:
    raise RuntimeError('No se detectó GPU. Selecciona T4 en el tipo de entorno y vuelve al paso 1.')
if not all(report['modules'].values()) or not all(report['commands'].values()):
    raise RuntimeError('Faltan componentes. Revisa la instalación del paso 1 antes de continuar.')
song_form = create_song_form()
print('Motor listo · GPU:', report['gpu'])


## 2 · Elegir canción
Ejecuta **solo una** opción: **2a** (desde tu dispositivo) o **2b** (YouTube).

Cuando veas **Archivo listo**, continúa con el paso 3.

In [ ]:
# @title 2a · Subir canción desde tu dispositivo
# @markdown Pulsa ▶ y selecciona tu audio o video.
if 'song_form' not in globals():
    raise RuntimeError('Primero completa el paso 1.')
song_form = create_song_form()
selected_background = {}
upload_dir = ROOT / 'uploads' / uuid.uuid4().hex
print('Selecciona un archivo y espera a que termine de subir…', flush=True)
uploaded = files.upload(target_dir=str(upload_dir))
uploaded_paths = [Path(name) for name in uploaded]
del uploaded
if len(uploaded_paths) != 1:
    raise ValueError('Selecciona exactamente un audio o video. Vuelve a ejecutar este paso.')
audio_path = uploaded_paths[0]
if audio_path.suffix.lower() not in {'.wav', '.flac', '.mp3', '.m4a', '.ogg', '.mp4', '.webm'}:
    raise ValueError('Formato no admitido. Usa WAV, FLAC, MP3, M4A, OGG, MP4 o WEBM.')
if not audio_path.is_file() or not audio_path.stat().st_size:
    raise ValueError('No se recibió un archivo con contenido. Vuelve a ejecutar este paso.')
song_form['state']['audio'] = audio_path
print(f'✓ Archivo recibido: {audio_path.name} · {audio_path.stat().st_size / 1024 / 1024:.1f} MB')
print('Archivo listo. Continúa con el paso 3.')


In [ ]:
# @title 2b · Descargar canción de YouTube
# @markdown Pega el enlace y pulsa ▶.
Enlace = '' # @param {type:'string'}
# @markdown **¿Falla o pide iniciar sesión?** Marca la casilla y ejecuta de nuevo.
# @markdown Al aparecer el botón, sube tu `cookies.txt` (formato Netscape).
Subir_cookies = False # @param {type:'boolean'}
if 'song_form' not in globals():
    raise RuntimeError('Primero completa el paso 1.')
if not Enlace.strip():
    raise ValueError('Pega el enlace de un video de YouTube antes de pulsar ▶.')
song_form = create_song_form()
selected_background = {}
upload_dir = ROOT / 'uploads' / uuid.uuid4().hex
private = ROOT / 'private-cookies' / uuid.uuid4().hex
cookies = None
try:
    if Subir_cookies:
        print('Sube un único cookies.txt y espera a que termine la transferencia…', flush=True)
        uploaded = files.upload(target_dir=str(private))
        paths = [Path(name) for name in uploaded]
        del uploaded
        if len(paths) != 1 or paths[0].suffix.lower() != '.txt' or not paths[0].stat().st_size:
            raise ValueError('Selecciona un único cookies.txt con contenido en formato Netscape.')
        cookies = str(paths[0])
        print('✓ Cookies recibidas. Iniciando descarga…', flush=True)
    if shutil.which('deno') is None:
        from urllib.request import urlretrieve
        tools_dir = ROOT / 'tools'
        tools_dir.mkdir(exist_ok=True)
        deno = tools_dir / 'deno'
        if not deno.is_file():
            print('Instalando Deno para descargar de YouTube…', flush=True)
            archive = tools_dir / 'deno.zip'
            urlretrieve('https://github.com/denoland/deno/releases/download/v2.7.1/'
                        'deno-x86_64-unknown-linux-gnu.zip', archive)
            with zipfile.ZipFile(archive) as package:
                deno.write_bytes(package.read('deno'))
            deno.chmod(0o755)
            archive.unlink()
        os.environ['PATH'] = str(tools_dir) + os.pathsep + os.environ['PATH']
    selection = run_engine('youtube', {'url': Enlace.strip(), 'directory': str(upload_dir),
                                       'cookies': cookies})
finally:
    shutil.rmtree(private, ignore_errors=True)
audio_path = Path(selection['source'])
if not audio_path.is_file() or not audio_path.stat().st_size:
    raise ValueError('No se recibió un archivo con contenido. Vuelve a ejecutar este paso.')
song_form['state'].update(audio=audio_path, source_metadata=selection['metadata'])
print(f'✓ Archivo recibido: {audio_path.name} · {audio_path.stat().st_size / 1024 / 1024:.1f} MB')
print('Archivo listo. Continúa con el paso 3.')


In [ ]:
# @title 3 · Datos de la canción
# @markdown Pulsa ▶ para detectar título, artista e idioma. Después corrige lo que haga falta en el formulario.
if 'song_form' not in globals() or song_form['state']['audio'] is None:
    raise RuntimeError('Primero sube un audio en el paso 2 y espera la confirmación.')
state = song_form['state']
state['validated_audio'] = None
print('Leyendo metadatos…', flush=True)
selection = run_engine('validate-input', {'paths': [str(state['audio'])]})
metadata = selection.get('metadata', {})
metadata.update({key: value for key, value in state.get('source_metadata', {}).items() if value})
for key, label in [('title', 'Título'), ('artist', 'Artista'), ('album', 'Álbum')]:
    field = song_form['fields'][key]
    if not field.value.strip() or field.value == state['metadata'].get(key):
        field.value = metadata.get(key) or ''
    print(f'{label} detectado: {metadata.get(key) or "Sin etiqueta; puedes completarlo manualmente"}')
state['metadata'] = metadata
language_key = (str(state['audio']), WHISPER_MODEL)
if not state['language_manual']:
    if state.get('language_key') != language_key:
        print('Detectando idioma con Whisper', WHISPER_MODEL, '…', flush=True)
        print('La primera vez descarga el modelo; se reutiliza en el paso 4.', flush=True)
        try:
            state['language_detection'] = run_engine('detect-language', {
                'source': str(state['audio']), 'whisper_model': WHISPER_MODEL,
                'whisper_cache': str(MODEL_CACHE),
            })
        except RuntimeError:
            state['language_detection'] = {'language': None, 'reason': 'detection_unavailable'}
        state['language_key'] = language_key
    detection = state['language_detection']
    state['setting_language'] = True
    try:
        song_form['fields']['language'].value = detection.get('language') or ''
    finally:
        state['setting_language'] = False
    if detection.get('language'):
        label = dict((code, label) for label, code in report['notebook_languages'])[detection['language']]
        print(f'Idioma sugerido: {label}. Puedes corregirlo en el formulario.')
    else:
        print('No se pudo determinar el idioma; selecciónalo manualmente')
else:
    print('Se conserva el idioma seleccionado manualmente.')
state['validated_audio'] = state['audio']
display(song_form['panel'])


In [ ]:
# @title 4 · Crear voces y letra sincronizada
# @markdown Pulsa ▶ para separar las voces y sincronizar la letra (usa tu TXT o la busca en línea).
# @markdown Al terminar verás de dónde salieron la letra y los tiempos.
if 'song_form' not in globals() or song_form['state']['audio'] is None:
    raise RuntimeError('Selecciona un audio en el paso 2.')
if song_form['state'].get('validated_audio') != song_form['state']['audio']:
    raise RuntimeError('Ejecuta el paso 3 para leer y confirmar los datos del audio actual.')
preparing_form = song_form
preparing_inputs = song_snapshot(preparing_form)
fields = song_form['fields']
audio = song_form['state']['audio']
reference = song_form['state']['reference']
TITULO = fields['title'].value
ARTISTA = fields['artist'].value
ALBUM = fields['album'].value
IDIOMA = fields['language'].value
if IDIOMA not in {code for _, code in report['notebook_languages']}:
    raise ValueError('Selecciona el idioma en el formulario del paso 3 antes de preparar.')
BUSCAR_LETRA_EN_LINEA = reference is None
if BUSCAR_LETRA_EN_LINEA and (not TITULO.strip() or not ARTISTA.strip()):
    raise ValueError('Completa título y artista en el formulario para buscar la letra automáticamente.')
print('Modelo Whisper:', WHISPER_MODEL, '· alineación acústica automática', flush=True)
print('Validando audio…', flush=True)
selection = run_engine('validate-input', {'paths': [str(audio)] + ([str(reference)] if reference else [])})
media_info = selection['media']
print(f"Audio validado: {audio.name} · {media_info['duration_seconds']:.1f} segundos")
print('Letra:', reference.name if reference else 'Búsqueda en línea')
next_prepared = run_engine('prepare', {
    'source': str(audio), 'output_dir': str(OUTPUT_DIR), 'title': TITULO.strip() or None,
    'artist': ARTISTA.strip() or None, 'album': ALBUM.strip() or None, 'language': IDIOMA,
    'lyrics_text': str(reference) if reference else None, 'use_lrclib': BUSCAR_LETRA_EN_LINEA,
    'whisper_model': WHISPER_MODEL, 'whisper_cache': str(MODEL_CACHE),
    'source_metadata': preparing_form['state'].get('source_metadata'),
})
if song_form is not preparing_form or song_snapshot(song_form) != preparing_inputs:
    raise RuntimeError(f'Cambiaste los datos durante el proceso. Resultado conservado en {next_prepared["directory"]}.')
prepared = next_prepared
rendered = None
preparing_form['state'].update(prepared_directory=prepared['directory'], prepared_snapshot=preparing_inputs)
preparing_form['refresh']()
print('Proyecto listo. Puedes corregir la letra (4b), elegir el fondo (4c) o ir directo al paso 5.')
info = run_engine('inspect', {'directory': prepared['directory']})
evidence = info['review']
sources = {'lrclib': 'LRCLIB', 'genius': 'Genius', 'txt': 'TXT aportado',
           'whisper': 'Transcripción automática (Whisper)'}
print('Origen de la letra:', sources.get(evidence['reference_source'], 'Sin información'))
print('Motor de sincronización:', evidence.get('alignment_engine') or 'Sin información')
if evidence.get('acoustic_verification') == 'unavailable':
    print('La comprobación independiente con Whisper no pudo completarse; revisa el diagnóstico.')
if evidence['word_timing'] == 'estimated':
    print('Algunos tiempos son estimados; requieren revisión.')
if evidence['reference_source'] == 'whisper':
    print('La transcripción automática puede contener errores.')
if evidence['needs_review'] or evidence['anchor_status'] == 'insufficient_anchors':
    print('Tiempos pendientes de revisión. Corrígelos en la sección 4b con LyricsEditor.')


## 4b · Corregir letra y tiempos · opcional
Ejecuta esta sección solo si quieres corregir la letra o los tiempos a mano.

**Descargar voces → Descargar letra → Editar → Subir corrección**

Si no necesitas corregir nada, salta al paso 4c o al 5.

In [ ]:
# @title 4b.1 · Descargar voces
# @markdown Pulsa ▶ para descargar `vocals.flac`.
if not globals().get('prepared'):
    raise RuntimeError('Primero completa el paso 4.')
project = prepared['directory']
info = run_engine('inspect', {'directory': project})
filename = Path(info['vocals'])
print(f'Descargando: {filename.name}')
files.download(str(filename))


In [ ]:
# @title 4b.2 · Descargar letra
# @markdown Pulsa ▶ para descargar `lyrics.lyricsfile.yaml`.
if not globals().get('prepared'):
    raise RuntimeError('Primero completa el paso 4.')
filename = Path(prepared['directory']) / 'lyrics.lyricsfile.yaml'
print(f'Descargando: {filename.name}')
files.download(str(filename))


### 4b.3 · Editar en LyricsEditor
**[Abrir LyricsEditor ↗](https://dotslashgabut.github.io/lyricseditor/)**

1. Abre `vocals.flac` y `lyrics.lyricsfile.yaml` en el editor.
2. Corrige el texto y los tiempos escuchando las voces.
3. Exporta como **LyricsFile YAML** y súbelo en el paso 4b.4.

In [ ]:
# @title 4b.4 · Subir letra corregida
# @markdown Pulsa ▶ y sube el YAML exportado desde LyricsEditor. Espera el mensaje **YAML actualizado**.
if not globals().get('prepared'):
    raise RuntimeError('Primero completa el paso 4.')
project = prepared['directory']
folder = ROOT / 'reviews' / uuid.uuid4().hex
print('Sube el YAML corregido…', flush=True)
uploaded = files.upload(target_dir=str(folder))
paths = [Path(name) for name in uploaded]
del uploaded
if len(paths) != 1 or paths[0].suffix.lower() not in {'.yaml', '.yml'}:
    raise ValueError('Selecciona exactamente un archivo LyricsFile YAML exportado por el editor.')
if not globals().get('prepared') or prepared['directory'] != project:
    raise RuntimeError('La canción cambió. Repite esta celda para el proyecto actual.')
run_engine('import-lyrics', {'directory': project, 'source': str(paths[0])})
rendered = None
print('✓ YAML actualizado. Continúa con el paso 4c o el 5.')


## 4c · Fondo del video · opcional
**Abrir controles → Elegir fondo → Aplicar.** Si lo omites, se usa el fondo Clásico.

Opciones: Clásico, Fluido, Cósmico, Color, Imagen, Video o Video de origen. Las vistas previas son un solo fotograma.

In [ ]:
# @title 4c.1 · Elegir fondo
# @markdown Pulsa ▶ y elige el fondo en el panel que aparece debajo. Luego ejecuta 4c.2.
ensure_current_song()

def create_design_form(project):
    """Create live visual controls scoped to one prepared song."""
    saved = run_engine('inspect', {'directory': project})['design']
    background = saved['background']
    initial = {'background': {'solid': 'Color', 'image': 'Imagen', 'video': 'Video'}.get(
        background['kind'], 'Cósmico' if background.get('appearance') == 'cosmic' else 'Fluido'),
        'color': background.get('color', '#0B1020'), 'overlay': background.get('overlay_opacity', 0.3)}
    previous = globals().get('selected_background', {})
    if previous.get('directory') == project and previous.get('saved_design') == saved:
        initial = previous['values'].copy()
    previews = ROOT / 'style-previews'
    previews.mkdir(exist_ok=True)
    preview_script = '''
import sys
from pathlib import Path
from types import SimpleNamespace
from karaoke_app.visual_config import preparation_visuals
from karaoke_app.render_graphics import background_frame
video = {'width': 320, 'height': 180, 'fps': 24}
for name in ('classic', 'fluid', 'cosmic'):
    visual = preparation_visuals(video, SimpleNamespace(preset=name))
    document = {'video': video, 'background': visual['render_background']}
    background_frame(Path('.'), document).save(Path(sys.argv[1]) / f'{name}.png')
'''
    run_command([str(ENGINE), '-c', preview_script, str(previews)])
    fields = {
        'background': widgets.Dropdown(description='Fondo', options=[
            'Clásico', 'Fluido', 'Cósmico', 'Color', 'Imagen', 'Video', 'Video de origen']),
        'color': widgets.ColorPicker(value='#0B1020', description='Personalizar', concise=False),
        'overlay': widgets.FloatSlider(value=0.3, min=0, max=1, step=0.05,
                                       description='Oscurecer', continuous_update=False),
    }
    palette = widgets.Dropdown(description='Paleta', options=[
        ('Azul noche', '#0B1020'), ('Negro', '#000000'), ('Grafito', '#242424'),
        ('Azul petróleo', '#17334D'), ('Verde bosque', '#143D2B'),
        ('Borgoña', '#4A1930'), ('Violeta', '#33204F'), ('Personalizado', None)])
    for key, value in initial.items():
        fields[key].value = value
    palette.value = initial['color'] if initial['color'] in {value for _, value in palette.options} else None
    for control in [*fields.values(), palette]:
        control.layout = widgets.Layout(width='100%', max_width='360px')
        control.style.description_width = '85px'
    swatch = widgets.HTML(layout=widgets.Layout(width='54px', flex='0 0 54px'))
    preview = widgets.Image(format='png', layout=widgets.Layout(width='240px', max_width='100%'))
    hint = widgets.HTML()
    status = widgets.HTML()
    palette_row = widgets.HBox([palette, swatch], layout=widgets.Layout(align_items='center', width='100%'))
    color_panel = widgets.VBox([palette_row, fields['color']])
    form = {'directory': project, 'fields': fields, 'palette': palette, 'status': status,
            'preview': preview, 'applied': initial.copy(), 'saved': saved}

    def values():
        return {key: field.value for key, field in fields.items()}

    def refresh(change=None):
        current = values()
        background = current['background']
        color_panel.layout.display = '' if background == 'Color' else 'none'
        fields['color'].layout.display = '' if palette.value is None else 'none'
        fields['overlay'].layout.display = (
            '' if background in {'Imagen', 'Video', 'Video de origen'} else 'none')
        color = escape(current['color'], quote=True)
        swatch.value = (f'<span title="{color}" style="display:inline-block;width:28px;height:28px;'
                        f'border:1px solid #888;border-radius:4px;background:{color}"></span>')
        preset = {'Clásico': 'classic', 'Fluido': 'fluid', 'Cósmico': 'cosmic'}.get(background)
        preview.layout.display = '' if preset else 'none'
        preview.value = (previews / f'{preset}.png').read_bytes() if preset else b''
        hints = {'Clásico': 'Azul noche liso.',
                 'Fluido': 'Fondo animado suave.',
                 'Cósmico': 'Fondo animado con textura.',
                 'Color': 'Un solo color a tu elección.',
                 'Imagen': 'Subirás tu imagen en 4c.2.',
                 'Video': 'Subirás tu video en 4c.2. Si es corto, se repite.',
                 'Video de origen': 'Usa el video de la canción. El audio no cambia.'}
        hint.value = f'<div style="white-space:normal"><small>{hints[background]}</small></div>'
        status.value = ('✓ Fondo guardado. Continúa al paso 5.' if form['applied'] == current
                        else 'Selección pendiente · ejecuta 4c.2 para aplicarla.')

    def choose_palette(change):
        if change['new'] is not None:
            fields['color'].value = change['new']
        refresh()

    palette.observe(choose_palette, names='value')
    for control in fields.values():
        control.observe(refresh, names='value')
    form['values'] = values
    form['refresh'] = refresh
    form['panel'] = widgets.VBox([
        fields['background'], preview, color_panel, fields['overlay'], hint, status,
    ], layout=widgets.Layout(width='100%', max_width='520px', gap='8px', padding='8px'))
    refresh()
    return form

if (globals().get('design_form', {}).get('directory') != prepared['directory']
        or 'preset' in globals().get('design_form', {}).get('fields', {})):
    design_form = create_design_form(prepared['directory'])
display(design_form['panel'])


In [ ]:
# @title 4c.2 · Aplicar fondo
# @markdown Pulsa ▶ para aplicar el fondo elegido. Con Imagen o Video, sube el archivo cuando aparezca el botón.
# @markdown Si cambias la selección, repite esta celda.
ensure_current_song()
project = prepared['directory']
if globals().get('design_form', {}).get('directory') != project:
    raise RuntimeError('Primero ejecuta 4c.1 y elige un fondo.')
chosen = design_form['values']()
Fondo, Color, Oscurecer = chosen['background'], chosen['color'], chosen['overlay']
unchanged = chosen == design_form['applied'] and Fondo not in {'Imagen', 'Video'}
options = {} if unchanged else {
    'preset': {'Clásico': 'classic', 'Fluido': 'fluid', 'Cósmico': 'cosmic'}.get(Fondo, 'classic')}
if not unchanged and Fondo == 'Color':
    options['background_color'] = Color
elif not unchanged and Fondo in {'Imagen', 'Video', 'Video de origen'}:
    folder = Path(project) / 'assets' / f'background-{uuid.uuid4().hex}'
    if Fondo == 'Video de origen':
        source = song_form['state']['audio']
        if source is None or not source.is_file() or source.suffix.lower() not in {'.mp4', '.webm'}:
            raise ValueError('No hay un video de origen disponible.')
        folder.mkdir(parents=True)
        background = folder / f'background{source.suffix.lower()}'
        shutil.copy2(source, background)
    else:
        print('Sube el archivo de fondo y espera a que termine…', flush=True)
        uploaded = files.upload(target_dir=str(folder))
        paths = [Path(name) for name in uploaded]
        del uploaded
        if len(paths) != 1:
            raise ValueError('Selecciona un solo archivo de fondo.')
        background = paths[0]
    key = 'background_image' if Fondo == 'Imagen' else 'background_video'
    options.update({key: str(background), 'background_overlay_opacity': Oscurecer})
if (globals().get('prepared') or {}).get('directory') != project or design_form['values']() != chosen:
    raise RuntimeError('La canción o el fondo cambió durante la subida. Repite 4c.2.')
if options:
    run_engine('configure', {'directory': project, 'options': options})
saved_design = run_engine('inspect', {'directory': project})['design']
selected_background = {'directory': project, 'options': options, 'label': Fondo,
                       'values': chosen.copy(), 'saved_design': saved_design}
if Fondo == 'Color':
    selected_background['label'] += f' {Color}'
design_form['applied'] = chosen
design_form['saved'] = saved_design
design_form['refresh']()
if options:
    rendered = None
print('✓ Fondo aplicado:', selected_background['label'], '· Continúa al paso 5.')


In [ ]:
# @title 5 · Generar el video
# @markdown Elige el tipo de video y la calidad, y pulsa ▶.
Modo = "Lyrics" # @param ["Lyrics", "Karaoke"]
# @markdown Lyrics conserva la voz original. Karaoke quita la voz para que tú cantes.
Calidad = '720p (rápido)' # @param ['720p (rápido)', '1080p (mayor calidad)']
ensure_current_song()
design = globals().get('design_form', {})
if design.get('directory') == prepared['directory'] and design['applied'] != design['values']():
    raise RuntimeError('Hay un fondo sin aplicar. Ejecuta 4c.2 antes de generar.')
options = {
    'mode': {
        'Lyrics': 'lyrics',
        'Karaoke': 'karaoke',
    }[Modo],
    'resolution': Calidad.split()[0],
}
background = globals().get('selected_background', {})
if background.get('directory') == prepared['directory']:
    print('Fondo:', background['label'])
else:
    print('Fondo: se conserva el guardado del proyecto.')
run_engine('configure', {'directory': prepared['directory'], 'options': options})
next_rendered = run_engine('render', {'directory': prepared['directory'], 'options': options})
rendered = next_rendered
print('Video listo. Ejecuta el paso 6 para descargarlo.')
for video in rendered['videos']:
    print(Path(video).name)
    display(Video(video, embed=True, width=640))


In [ ]:
# @title 6 · Descargar
# @markdown Pulsa ▶ para descargar el video en MP4.
if not globals().get('rendered'):
    raise RuntimeError('Primero completa el paso 5.')
for filename in rendered['videos']:
    print('Descargando:', Path(filename).name)
    files.download(filename)

## Al terminar
Comprueba que el MP4 se abre en tu dispositivo. Después usa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** para liberar la GPU.

**Si algo falla:** lee el último mensaje y no sigas con los pasos siguientes. Si pierdes la sesión, vuelve al paso 1.